# 📘 Notebook 09: Text Classification with Naive Bayes

### Course: *From Words to Meaning: Natural Language Processing in Python*
**Instructor:** Ioannis Tsioulis

*Module C: Counting Words · Notebook 3 of 3 in this module · (9 of 18 overall)*

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Define **text classification** and give examples of where it is used
- Explain the **Naive Bayes** classifier, including the assumption that makes it "naive"
- Classify a short text **by hand** with Bayes' rule, logarithms and smoothing
- Implement Naive Bayes from scratch and train it on 2,000 film reviews
- Inspect a trained model to see **which words it relies on**
- Build the same classifier as a scikit-learn **pipeline**, compare it with logistic regression, and analyse its errors

> **Prerequisites:** Notebooks 07 and 08. Classification, the train/test split, the confusion matrix and cross-validation are explained in *From Python to Deep Learning & Fine-Tuning*, Notebooks 08, 10 and 11. Here we apply them to text.
>
> 🔭 **Why this notebook matters:** sorting text into categories is the most widely deployed task in NLP. It filters your spam, routes customer messages to the right team, flags abusive posts, and tells a company whether its reviews are happy or angry. Naive Bayes is the classic method: a few lines of counting that are hard to beat on small data, train in a fraction of a second, and can explain every decision they make.

> ℹ️ **Setup note.** Run this cell once. It downloads 2,000 film reviews.

In [ ]:
import nltk
nltk.download("movie_reviews", quiet=True)
nltk.download("reuters", quiet=True)

import math
import random
from collections import Counter

## 1. The task

### Intuition first
You glance at an e-mail and know at once that it is spam. How? Certain words give it away: *winner*, *free*, *urgent*, *click*. No single word is proof, but each one shifts your belief, and together they settle the matter. A text classifier does the same: it weighs the evidence of the words.

### Formal definition
**Text classification** assigns to a document $d$ one label $c$ from a fixed set of classes. Typical examples:

| Task | Classes |
|---|---|
| spam filtering | spam, not spam |
| **sentiment analysis** | positive, negative |
| topic labelling | sport, politics, business ... |
| language identification | English, Greek, French ... |

The classifier is **trained** on documents whose labels are known, and is then asked to label new ones. This is supervised learning. Our running example in this notebook is sentiment analysis of film reviews.

## 2. Naive Bayes

### Intuition first
Suppose that in reviews you have read before, the word *boring* turned up far more often in negative reviews than in positive ones. If a new review contains *boring*, that is evidence for "negative". If it also contains *predictable* and *waste*, the evidence piles up. Naive Bayes makes this precise: for each class, it asks *"how likely would these words be, if the review belonged to this class?"* and chooses the class under which the words are least surprising.

That question should sound familiar. It is the question a **language model** answers. Naive Bayes trains one unigram language model per class, and asks which of them finds the document most probable.

### Formal definition
By **Bayes' rule**, the probability of class $c$ given document $d$ is

$$P(c \mid d) = \frac{P(c)\, P(d \mid c)}{P(d)}$$

The denominator is the same for every class, so to find the best class we only need the numerator. A document is a sequence of words $w_1 \dots w_n$, and here comes the **naive assumption**: given the class, the words are independent of one another. Then

$$\hat{c} = \arg\max_c \; P(c) \prod_{i=1}^{n} P(w_i \mid c)$$

- $P(c)$, the **prior**: the share of training documents in class $c$.
- $P(w \mid c)$, the **likelihood**: how often word $w$ occurs among all the words of the documents of class $c$.

As in Notebook 07, we use **add-one smoothing** so that a word never seen in a class does not turn the whole product into zero, and we add **logarithms** in place of multiplying tiny numbers:

$$P(w \mid c) = \frac{\text{count}(w, c) + 1}{\text{count}(c) + V} \qquad\qquad \text{score}(c) = \log P(c) + \sum_i \log P(w_i \mid c)$$

The assumption is called naive because it is plainly false: after *New*, the word *York* is far from independent. The method works well anyway, because to choose the right class the probabilities need not be accurate. They only need to point the right way.

## 3. A classification by hand

Five tiny reviews are enough to follow every step.

In [ ]:
small_training_set = [
    ("a great film with a great cast", "pos"),
    ("wonderful and moving", "pos"),
    ("great fun", "pos"),
    ("a boring and predictable film", "neg"),
    ("terrible acting and a boring plot", "neg"),
]

class_counts = Counter()
word_counts = {"pos": Counter(), "neg": Counter()}
vocabulary = set()

for text, label in small_training_set:
    class_counts[label] += 1
    word_counts[label].update(text.split())
    vocabulary.update(text.split())

V = len(vocabulary)
print("Documents per class:", dict(class_counts))
print("Words per class:    ", {label: sum(counts.values()) for label, counts in word_counts.items()})
print("Vocabulary size:    ", V)

Now classify the new review *"a boring film"*. For each class we start from the prior and multiply by the smoothed likelihood of each word.

In [ ]:
new_review = "a boring film".split()

for label in ["pos", "neg"]:
    prior = class_counts[label] / sum(class_counts.values())
    total_words = sum(word_counts[label].values())
    print(f"Class {label}: prior = {class_counts[label]}/{sum(class_counts.values())} = {prior:.2f}")

    probability = prior
    for word in new_review:
        likelihood = (word_counts[label][word] + 1) / (total_words + V)
        print(f"   P({word} | {label}) = ({word_counts[label][word]} + 1) / ({total_words} + {V}) = {likelihood:.3f}")
        probability *= likelihood
    print(f"   prior x likelihoods = {probability:.6f}\n")

The score for `neg` is more than twice the score for `pos`, so the review is classified as negative. Trace where the difference comes from: *a* and *film* are about equally likely in both classes, and contribute almost nothing to the decision. The word *boring* occurs twice in the negative reviews and never in the positive ones, and that settles it. Thanks to smoothing, its probability under `pos` is small and not zero.

## 4. Naive Bayes from scratch

The same steps as two functions. A document is a list of words. Training is nothing but counting.

In [ ]:
def train_naive_bayes(examples):
    class_counts = Counter()
    word_counts = {}
    vocabulary = set()
    for words, label in examples:
        class_counts[label] += 1
        if label not in word_counts:
            word_counts[label] = Counter()
        word_counts[label].update(words)
        vocabulary.update(words)
    word_totals = {label: sum(counts.values()) for label, counts in word_counts.items()}
    return class_counts, word_counts, word_totals, vocabulary

def class_scores(words, model, k=1):
    class_counts, word_counts, word_totals, vocabulary = model
    documents = sum(class_counts.values())
    scores = {}
    for label in class_counts:
        score = math.log(class_counts[label] / documents)              # log prior
        denominator = word_totals[label] + k * len(vocabulary)
        for word in words:
            if word in vocabulary:                                     # ignore words never seen in training
                score += math.log((word_counts[label][word] + k) / denominator)
        scores[label] = score
    return scores

def classify(words, model, k=1):
    scores = class_scores(words, model, k)
    return max(scores, key=scores.get)

toy_model = train_naive_bayes([(text.split(), label) for text, label in small_training_set])

for text in ["a boring film", "great cast and a moving plot", "the plot"]:
    print(f"{text:<30} {classify(text.split(), toy_model)}")

### Real data: 2,000 film reviews
The corpus contains 1,000 positive and 1,000 negative reviews written by film fans, already tokenised and lower-cased. We keep the alphabetic tokens, shuffle, and hold out 400 reviews for testing.

In [ ]:
from nltk.corpus import movie_reviews

reviews = []
for label in ["pos", "neg"]:
    for file_id in movie_reviews.fileids(label):
        words = [word for word in movie_reviews.words(file_id) if word.isalpha()]
        reviews.append((words, label, movie_reviews.raw(file_id)))

random.seed(1)
random.shuffle(reviews)
train, test = reviews[:1600], reviews[1600:]

words, label, raw = train[0]
print(len(reviews), "reviews. Average length:", sum(len(r[0]) for r in reviews) // len(reviews), "words.\n")
print(label, "|", " ".join(words[:60]), "...")

In [ ]:
model = train_naive_bayes([(words, label) for words, label, raw in train])

correct = 0
for words, label, raw in test:
    if classify(words, model) == label:
        correct += 1

accuracy = correct / len(test)
print(f"Vocabulary: {len(model[3]):,} words")
print(f"Accuracy on 400 unseen reviews: {accuracy:.1%}")

The two classes are the same size, so a classifier that guessed at random, or always answered "positive", would score 50%. Our forty lines of counting do very much better than that, on full-length reviews written in free and often sarcastic prose.

## 5. Looking inside the model

A great strength of Naive Bayes is that it is easy to inspect. Which words push hardest towards each class? For each word, compare its probability in positive reviews with its probability in negative reviews. The logarithm of that ratio is positive for words that favour "pos" and negative for words that favour "neg".

In [ ]:
class_counts, word_counts, word_totals, vocabulary = model

def log_ratio(word):
    p_pos = (word_counts["pos"][word] + 1) / (word_totals["pos"] + len(vocabulary))
    p_neg = (word_counts["neg"][word] + 1) / (word_totals["neg"] + len(vocabulary))
    return math.log(p_pos / p_neg)

reviews_containing = Counter()                    # in how many training reviews does each word appear?
for words, label, raw in train:
    reviews_containing.update(set(words))

widespread = [word for word in vocabulary if reviews_containing[word] >= 60]
ranked = sorted(widespread, key=log_ratio)

print("Strongest evidence for NEGATIVE:")
print("  ", ranked[:14])
print("Strongest evidence for POSITIVE:")
print("  ", ranked[-14:][::-1])

The model was never told which words are positive or negative. It found them by counting. We looked only at words that appear in at least 60 reviews. Lower that number to 10 and run the cell again: the lists fill up with the names of actors and films that happened to be loved or hated in this particular collection. The classifier has no idea what a word means. It knows only which class a word **keeps company with**, and it will happily learn an accident of the data as if it were a law.

> ⚠️ **Common pitfalls**
>
> - Trusting a classifier on text unlike its training data. A model trained on film reviews has learned that *unpredictable* is praise. In a review of a car's brakes it is not.
> - Forgetting the prior. With balanced classes it changes nothing. With 95% of e-mails legitimate, it rightly makes the classifier cautious about calling something spam.
> - Forgetting to smooth. Without it, a single word unseen in one class gives that class probability zero, whatever the other words say.

## 6. The same model as a scikit-learn pipeline

In practice we use a library. A **pipeline** chains the vectoriser of Notebook 08 with a classifier, so that raw text goes in at one end and labels come out at the other.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import confusion_matrix, classification_report

train_texts = [raw for words, label, raw in train]
train_labels = [label for words, label, raw in train]
test_texts = [raw for words, label, raw in test]
test_labels = [label for words, label, raw in test]

naive_bayes = make_pipeline(CountVectorizer(), MultinomialNB())
naive_bayes.fit(train_texts, train_labels)

print(f"scikit-learn Naive Bayes: {naive_bayes.score(test_texts, test_labels):.1%}")
print(f"Our own Naive Bayes:      {accuracy:.1%}")

The two agree closely, as they should: it is the same algorithm. Any small difference comes from the tokeniser.

Accuracy is a single number. The **confusion matrix** shows which mistakes are made, and the report gives precision and recall for each class.

In [ ]:
predictions = naive_bayes.predict(test_texts)

print("Rows: true class. Columns: predicted class. Order: neg, pos\n")
print(confusion_matrix(test_labels, predictions, labels=["neg", "pos"]))
print()
print(classification_report(test_labels, predictions))

### A stronger model, and an honest comparison
Naive Bayes is one way to weigh words. **Logistic regression** is another: it learns a weight for each word directly, and does not assume the words are independent. Combined with TF-IDF it is the standard strong baseline for text classification.

A test set of 400 reviews is small, and a difference of one or two points between two models may be luck. **Cross-validation** gives a steadier comparison: the data are split five ways, and each model is trained and tested five times.

In [ ]:
from sklearn.model_selection import cross_val_score

all_texts = [raw for words, label, raw in reviews]
all_labels = [label for words, label, raw in reviews]

candidates = {
    "Naive Bayes, counts":          make_pipeline(CountVectorizer(), MultinomialNB()),
    "Logistic regression, TF-IDF":  make_pipeline(TfidfVectorizer(sublinear_tf=True), LogisticRegression(max_iter=1000)),
}

for name, pipeline in candidates.items():
    scores = cross_val_score(pipeline, all_texts, all_labels, cv=5)
    print(f"{name:<30} {scores.mean():.1%}   (five runs: {', '.join(f'{s:.0%}' for s in scores)})")

Look at the spread of the five runs before you look at the averages. The same model gives results that differ by several points from one split to another. A difference between two models is only worth reporting when it is clearly larger than that spread.

## 7. Where it fails, and why

Let us read some of the mistakes. A model's errors are more instructive than its accuracy.

In [ ]:
shown = 0
for text, truth, predicted in zip(test_texts, test_labels, predictions):
    if truth != predicted and len(text) < 2500:
        print(f"TRUE: {truth}   PREDICTED: {predicted}")
        print(" ".join(text.split())[:420], "...\n")
        shown += 1
        if shown == 2:
            break

Read them and ask why a word-counter would be misled. Common reasons are a positive review that spends most of its length describing a grim plot, a negative review that lists what the film *could* have been, and irony. All have the same root: the meaning is in how the words are **combined**, and the bag of words has thrown the combination away.

The plainest case is negation.

In [ ]:
for text in ["this film is good", "this film is not good", "this film is not bad at all"]:
    probabilities = naive_bayes.predict_proba([text])[0]
    print(f"{text:<30} P(neg) = {probabilities[0]:.2f}   P(pos) = {probabilities[1]:.2f}")

The first two sentences receive almost the same probabilities: adding *not* changes nothing, because *not* is equally common in both kinds of review. And *"not bad at all"*, which is a compliment, is judged clearly negative, because it contains *bad*. To a bag of words, the words are what counts, and never the way they are combined. A partial repair is to count **pairs of words** as well as single words, so that *not good* becomes a feature of its own. This brings back a little of the word order that Notebook 07 exploited.

In [ ]:
with_pairs = make_pipeline(CountVectorizer(ngram_range=(1, 2), min_df=2), MultinomialNB())

scores_single = cross_val_score(make_pipeline(CountVectorizer(min_df=2), MultinomialNB()), all_texts, all_labels, cv=5)
scores_pairs = cross_val_score(with_pairs, all_texts, all_labels, cv=5)

print(f"Single words:          {scores_single.mean():.1%}")
print(f"Single words + pairs:  {scores_pairs.mean():.1%}")

Pairs help a little, at the cost of a far larger and sparser vocabulary. That is as far as counting will take us. Here is the score for Module C.

| | N-gram model | Bag of words + TF-IDF | Naive Bayes on word counts |
|---|---|---|---|
| Turns text into numbers | yes | yes | yes |
| Makes a decision about a document | no | no | yes |
| Uses word order | a few words back | no | no (a little, with pairs) |
| Knows that two words are similar | no | no | no |

The last row has stayed the same for three notebooks. A review that says *superb* teaches the classifier nothing about *magnificent*, which it may never have seen. Every word is still an isolated symbol. The next module changes that.

---
## ✏️ Exercises

### Exercise 1 (By hand)
Using the five-review training set of Section 3, compute on paper the two scores for the review *"great plot"*, with add-one smoothing. Which class wins? Check with `class_scores`. (It returns logarithms: the larger one wins.)

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
# pos: prior 3/5.  P(great|pos) = (3+1)/(12+14) = 0.154   P(plot|pos) = (0+1)/(12+14) = 0.038
# neg: prior 2/5.  P(great|neg) = (0+1)/(11+14) = 0.040   P(plot|neg) = (1+1)/(11+14) = 0.080
print("pos:", 3/5 * 4/26 * 1/26)
print("neg:", 2/5 * 1/25 * 2/25)

scores = class_scores("great plot".split(), toy_model)
print(scores)
print({label: math.exp(score) for label, score in scores.items()})
```

*Positive wins: "great" was seen three times in positive reviews, which outweighs the single negative "plot". Taking the exponential of the log scores gives back the products computed by hand.*
</details>

### Exercise 2 (How much smoothing?)
`classify` has a parameter `k`, the amount added to every count. Compute the test accuracy of our own classifier for `k` in `[0.01, 0.1, 1, 10, 100]`. How sensitive is the result?

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
for k in [0.01, 0.1, 1, 10, 100]:
    correct = 0
    for words, label, raw in test:
        if classify(words, model, k) == label:
            correct += 1
    print(f"k = {k:<6} accuracy {correct / len(test):.1%}")
```

*The accuracy changes only slightly over a wide range of k, and falls when k is so large that the counts are drowned out. Compare this with Notebook 07, where k changed the perplexity enormously. A classifier only needs to rank two classes correctly, which is a much more forgiving job than estimating probabilities.*
</details>

### Exercise 3 (Do stop words matter here?)
Compare, with five-fold cross-validation, a Naive Bayes pipeline that removes English stop words (`CountVectorizer(stop_words="english")`) with one that keeps them. Before you run it: what do you expect, given what Notebook 03 said about negation?

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
keep = make_pipeline(CountVectorizer(), MultinomialNB())
remove = make_pipeline(CountVectorizer(stop_words="english"), MultinomialNB())

print(f"Stop words kept:    {cross_val_score(keep, all_texts, all_labels, cv=5).mean():.1%}")
print(f"Stop words removed: {cross_val_score(remove, all_texts, all_labels, cv=5).mean():.1%}")
```

*The difference is small, well within the spread between runs. Removing stop words is no magic ingredient: Naive Bayes already gives little weight to words that are equally common in both classes. Measure before you assume that a preprocessing step helps.*
</details>

### Exercise 4 (Present or absent?)
For opinions, *whether* a word appears may matter more than *how many times*. `CountVectorizer(binary=True)` records only presence. Compare it with ordinary counts using five-fold cross-validation.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
counts = make_pipeline(CountVectorizer(), MultinomialNB())
presence = make_pipeline(CountVectorizer(binary=True), MultinomialNB())

print(f"Counts:   {cross_val_score(counts, all_texts, all_labels, cv=5).mean():.1%}")
print(f"Presence: {cross_val_score(presence, all_texts, all_labels, cv=5).mean():.1%}")
```

*Presence usually does as well as counts for sentiment, and often slightly better. A reviewer who writes "bad" five times while describing the villain is not five times as negative. This variant is known as binary Naive Bayes.*
</details>

### Exercise 5 (How much data do we need?)
Train our own Naive Bayes on the first 50, 100, 200, 400, 800 and 1,600 training reviews, and print the test accuracy each time.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
for size in [50, 100, 200, 400, 800, 1600]:
    smaller_model = train_naive_bayes([(words, label) for words, label, raw in train[:size]])
    correct = 0
    for words, label, raw in test:
        if classify(words, smaller_model) == label:
            correct += 1
    print(f"{size:>5} reviews: {correct / len(test):.1%}")
```

*Even a hundred reviews give a result well above chance, and the gains shrink as the data grows. Naive Bayes has very few assumptions to learn, which is exactly why it is a good choice when labelled data is scarce.*
</details>

### Exercise 6 (A different task: topics, a little harder)
Naive Bayes is not tied to sentiment. From the Reuters corpus, take the articles that carry **exactly one** of the labels `crude`, `grain`, `trade` and `interest`. The file ids beginning with `training/` are the training set, and those beginning with `test/` the test set. Train a pipeline and report its accuracy and confusion matrix.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
from nltk.corpus import reuters

topics = ["crude", "grain", "trade", "interest"]
topic_train_texts, topic_train_labels, topic_test_texts, topic_test_labels = [], [], [], []

for file_id in reuters.fileids():
    labels = [label for label in reuters.categories(file_id) if label in topics]
    if len(labels) != 1:
        continue
    if file_id.startswith("training/"):
        topic_train_texts.append(reuters.raw(file_id))
        topic_train_labels.append(labels[0])
    else:
        topic_test_texts.append(reuters.raw(file_id))
        topic_test_labels.append(labels[0])

topic_model = make_pipeline(CountVectorizer(stop_words="english"), MultinomialNB())
topic_model.fit(topic_train_texts, topic_train_labels)

print(len(topic_train_texts), "training articles,", len(topic_test_texts), "test articles")
print(f"Accuracy: {topic_model.score(topic_test_texts, topic_test_labels):.1%}\n")
print(confusion_matrix(topic_test_labels, topic_model.predict(topic_test_texts), labels=topics))
```

*Topic classification is much easier than sentiment: the accuracy is far higher, with four classes in place of two. Topics are announced by their vocabulary, which is exactly what a bag of words captures. Opinions are expressed through how words are combined, which it does not.*
</details>

## 🔑 Key takeaways

- **Text classification** assigns a label to a document, after training on labelled examples.
- **Naive Bayes** picks the class under which the document's words are most probable: prior times the product of word likelihoods.
- The **naive** assumption (words are independent given the class) is false and works well anyway.
- Training is **counting**. Use **logarithms** and **smoothing**.
- The model can be **inspected**: the ratio of a word's probabilities shows which class it supports.
- Compare models with **cross-validation**, and look at the spread between runs before believing a small difference.
- Read the **errors**. A bag of words cannot handle negation, irony or anything else that depends on how words combine.

### 🏁 You have completed Module C: Counting Words!
With nothing more than counts you have built a language model that writes, a search engine, and a classifier. You have also met the wall that counting runs into three times in a row: every word is treated as a symbol unrelated to all the others.

**Next:** *Module D: Meaning as Vectors*, starting with *Notebook 10: Word Embeddings*, where words themselves become points in space, and *superb* finally ends up next to *magnificent*.

---
*End of Module C.*

---
*© Ioannis Tsioulis. *From Words to Meaning: Natural Language Processing in Python*. Prepared for educational use.*